In [1]:
from pathlib import Path
import pandas as pd

In [2]:
ROOT = Path.cwd().parent 
df = pd.read_stata(ROOT / "data" / "raw" / "brexitevote-ep2017.dta")
df = df[df["Pct_Leave"].notna()].rename(columns={"code": "Area_Code"}).reset_index(drop=True)

In [3]:
df

,id,Region_Code,Region,Area_Code,Area,Electorate,Pct_Turnout,Votes_Cast,Valid_Votes,Remain,...,rr_9,rr_10,rr_11,rr_12,cc_1,cc_2,cc_3,cc_4,anyrain23,anyrain22
0,70.0,E12000001,North East,E06000001,Hartlepool,70341.0,65.59,46134.0,46100.0,14029.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
1,71.0,E12000001,North East,E06000002,Middlesbrough,94612.0,64.89,61393.0,61358.0,21181.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
2,72.0,E12000001,North East,E06000003,Redcar and Cleveland,103529.0,70.26,72741.0,72714.0,24586.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
3,73.0,E12000001,North East,E06000004,Stockton-on-Tees,141486.0,71.00,100460.0,100415.0,38433.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
4,74.0,E12000001,North East,E06000005,Darlington,77662.0,71.07,55195.0,55166.0,24172.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
375,144.0,W92000004,Wales,W06000020,Torfaen,68957.0,69.86,48172.0,48144.0,19363.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
376,145.0,W92000004,Wales,W06000021,Monmouthshire,71607.0,77.74,55670.0,55630.0,28061.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
377,146.0,W92000004,Wales,W06000022,Newport,104977.0,70.21,73708.0,73649.0,32413.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
378,132.0,W92000004,Wales,W06000023,Powys,103270.0,77.00,79515.0,79469.0,36762.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0


In [4]:
for col in df.columns:
    print(col)

id
Region_Code
Region
Area_Code
Area
Electorate
Pct_Turnout
Votes_Cast
Valid_Votes
Remain
Leave
Pct_Remain
Pct_Leave
Pct_Rejected
EU75Leaveshare
pensionergrowth20012011
ResidentAge30to44share
ResidentAge45to59share
ResidentAge60plusshare
median_hourly_pay2005
median_hourly_pay_growth
mean_hourly_pay2005
mean_hourly_pay_growth
iqrpay2005
EU_2001Migrantshare
NONEU_2001Migrantshare
EU_Migrantgrowth
NONEU_Migrantgrowth
unqualifiedsharechange
self_employment_rate_aps
umemployment_rate_aps
economic_activity_rate_aps2015
MEP_Turnout
MEP_ConPct
MEP_LabPct
MEP_LDPct
MEP_UKIPPct
MEP_GreenPct
MEP_BNPPct
Total_EconomyEU_dependence
TotalImpactFLWAAYR
TREATED_WITHIN_62_DAYS
PRCP_AMT22
PRCP_AMT23
life_sat_mean
life_sat_cv
BREXITINDEXNARROW
noqual_2001_sh
quall1_2001_sh
quall2_2001_sh
quall3_2001_sh
quall4plus_2001_sh
noqual_sh_growth
quall1_sh_growth
quall2_sh_growth
quall3_sh_growth
quall4plus_sh_growth
ind2001_Agriculture_sh
ind2001_Mining_sh
ind2001_Manufacturing_sh
ind2001_Construction_sh
ind2001

## BIC / EBIC model selection for the Leave share

**Model.** `Pct_Leave` is an area-level *share*, not a 0/1 outcome, so the logistic model is put on the share itself:

$$\operatorname{logit}(p_i) = \beta_0 + x_{i,S}^\top \beta_S + \varepsilon_i,\qquad \varepsilon_i \sim N(0,\sigma^2),\qquad p_i = \text{Leave}_i / \text{Valid\_Votes}_i$$

That gives a proper likelihood with $n$ = number of areas, so BIC and EBIC are well defined.
(A binomial GLM on raw vote counts would treat ~33M votes as independent trials. It is massively overdispersed, and BIC would keep almost every covariate.)

**Criteria.** With $k = |S|$ selected covariates out of $p$ candidates:

$$\mathrm{EBIC}_\gamma(S) = -2\,\ell(\hat\beta_S,\hat\sigma^2) + (k+2)\log n + 2\gamma \log\binom{p}{k}$$

$\gamma = 0$ is ordinary BIC. Chen & Chen (2008) show EBIC is selection-consistent when $p = O(n^\kappa)$ and $\gamma > 1 - \tfrac{1}{2\kappa}$. Plain BIC is only consistent for fixed $p$, and with $p$ close to $n$ it tends to over-select.

**Search.** Best-subset search over $2^p$ models is infeasible, so candidate supports come from two paths: the lasso path and forward stepwise.
Each support is **refit by unpenalised OLS**, and the one with the lowest criterion wins.


In [5]:
import numpy as np
from scipy.special import logit, gammaln
from sklearn.linear_model import lasso_path
import statsmodels.api as sm

### 1. Outcome and candidate covariates
Excluded: identifiers, referendum-day quantities that mechanically contain the outcome, and the region (`rr_*`) and country (`cc_*`) dummies. Edit `EXCLUDE` / `GEO_DUMMY_PREFIXES` to change this.

In [6]:
IDENTIFIERS = ["id", "Region_Code", "Region", "Area_Code", "Area"]
OUTCOME_DERIVED = [           # same-day referendum quantities -> leakage
    "Pct_Leave", "Pct_Remain", "Leave", "Remain",
    "Votes_Cast", "Valid_Votes", "Pct_Rejected", "Pct_Turnout", "Electorate",
]
GEO_DUMMY_PREFIXES = ("rr_", "cc_")   # region and country dummies
EXCLUDE = IDENTIFIERS + OUTCOME_DERIVED

candidates = [c for c in df.select_dtypes("number").columns
              if c not in EXCLUDE and not c.startswith(GEO_DUMMY_PREFIXES)]
print(f"{len(candidates)} numeric candidate covariates")

80 numeric candidate covariates


### 2. One fixed estimation sample
Every model compared by BIC must be fit on **the same rows**. Otherwise the likelihoods are not comparable.
Columns with too many missing values are dropped first. Then the analysis keeps complete cases on the remaining columns.

In [7]:
MAX_MISSING = 0.05   # drop a covariate if more than 5% of areas are missing it

miss = df[candidates].isna().mean().sort_values(ascending=False)
print("Missing share (columns with any missing):")
print(miss[miss > 0].round(3).to_string() or "  none")

cols = miss[miss <= MAX_MISSING].index.tolist()
print(f"\nDropped for missingness (> {MAX_MISSING:.0%}): {sorted(set(candidates) - set(cols))}")

data = df[["Area", "Pct_Leave"] + cols].dropna().reset_index(drop=True)
print(f"Estimation sample: {len(data)} of {len(df)} areas")

# drop covariates that are constant in this sample (e.g. a region dummy whose region fell out)
const = [c for c in cols if data[c].nunique() <= 1]
cols = [c for c in cols if c not in const]
print(f"Dropped as constant: {const}")

Missing share (columns with any missing):
BREXITINDEXNARROW                 0.513
iqrpayGrowth                      0.034
eufundspercapitapayment13         0.029
iqrpay2005                        0.024
anyrain22                         0.011
anyrain23                         0.011
CommutingToLondon                 0.011
PRCP_AMT22                        0.011
PRCP_AMT23                        0.011
umemployment_rate_aps             0.008
self_employment_rate_aps          0.005
life_sat_mean                     0.005
life_sat_cv                       0.005
economic_activity_rate_aps2015    0.003
TotalImpactFLWAAYR                0.003
TREATED_WITHIN_62_DAYS            0.003

Dropped for missingness (> 5%): ['BREXITINDEXNARROW']
Estimation sample: 350 of 380 areas
Dropped as constant: []


### 3. Remove exactly collinear columns
Columns that are exact linear combinations of others (together with the intercept) make the OLS refit singular.
This step keeps columns in order and drops any column that adds no rank. Near-collinear blocks (for example, qualification shares that sum to about 1) are kept, but EBIC will rarely pick several of them at once.

In [8]:
def drop_exact_collinear(X: pd.DataFrame, tol: float = 1e-8):
    Z = (X - X.mean()) / X.std(ddof=0)
    basis = np.ones((len(Z), 1))
    kept, dropped = [], []
    for c in Z.columns:
        trial = np.column_stack([basis, Z[c].to_numpy()])
        if np.linalg.matrix_rank(trial, tol=tol * np.sqrt(len(Z))) > basis.shape[1]:
            basis, kept = trial, kept + [c]
        else:
            dropped.append(c)
    return kept, dropped

cols, collinear = drop_exact_collinear(data[cols])
print(f"Dropped as exact linear combinations of earlier columns: {collinear}")

X_raw = data[cols]
Xs = ((X_raw - X_raw.mean()) / X_raw.std(ddof=0)).to_numpy()   # standardised
y = logit(data["Pct_Leave"].to_numpy() / 100)                  # logit of the Leave share
n, p = Xs.shape

kappa = np.log(p) / np.log(n)
gamma_min = max(0.0, 1 - 1 / (2 * kappa))
print(f"n = {n}, p = {p}, kappa = log p / log n = {kappa:.3f}")
print(f"Chen & Chen consistency needs gamma > {gamma_min:.3f}")

Dropped as exact linear combinations of earlier columns: ['PublicEmployment2014']
n = 350, p = 78, kappa = log p / log n = 0.744
Chen & Chen consistency needs gamma > 0.328


### 4. Criteria, candidate paths and selection

In [9]:
def rss_of(Xs, y, support):
    A = np.column_stack([np.ones(len(y)), Xs[:, list(support)]])
    beta, *_ = np.linalg.lstsq(A, y, rcond=None)
    r = y - A @ beta
    return float(r @ r)

def ebic(rss, n, k, p, gamma):
    loglik = -0.5 * n * (np.log(2 * np.pi * rss / n) + 1)        # Gaussian, sigma^2 = RSS/n
    log_binom = gammaln(p + 1) - gammaln(k + 1) - gammaln(p - k + 1)
    return -2 * loglik + (k + 2) * np.log(n) + 2 * gamma * log_binom

def lasso_supports(Xs, y, max_k, n_alphas=300):
    yc = y - y.mean()
    a_max = np.abs(Xs.T @ yc).max() / len(y)
    alphas = np.geomspace(a_max, a_max * 1e-4, n_alphas)
    _, coefs, _ = lasso_path(Xs, yc, alphas=alphas, max_iter=50_000)
    out = {frozenset(np.flatnonzero(np.abs(coefs[:, j]) > 1e-10)) for j in range(coefs.shape[1])}
    return {s for s in out if len(s) <= max_k}

def forward_supports(Xs, y, max_k):
    # Greedy forward selection: at each step add the column with the largest RSS reduction
    Q = np.ones((len(y), 1)) / np.sqrt(len(y))           # orthonormal basis of current model
    r = y - Q @ (Q.T @ y)
    support, out = [], {frozenset()}
    for _ in range(max_k):
        Xp = Xs - Q @ (Q.T @ Xs)                           # columns residualised on the model
        norms = (Xp ** 2).sum(axis=0)
        gain = np.where(norms > 1e-10, (r @ Xp) ** 2 / np.maximum(norms, 1e-300), -np.inf)
        gain[support] = -np.inf
        j = int(np.argmax(gain))
        if not np.isfinite(gain[j]):
            break
        q = Xp[:, j] / np.sqrt(norms[j])
        Q = np.column_stack([Q, q]); r = r - q * (q @ r)
        support.append(j); out.add(frozenset(support))
    return out

def select_model(Xs, y, gammas, max_k=None):
    # Return {gamma: best support} and the full table of evaluated candidate models
    n, p = Xs.shape
    max_k = max_k or min(p, n // 5)
    supports = lasso_supports(Xs, y, max_k) | forward_supports(Xs, y, max_k)
    rows = []
    for s in supports:
        rss = rss_of(Xs, y, s)
        row = {"support": s, "k": len(s), "rss": rss}
        for g in gammas:
            row[g] = ebic(rss, n, len(s), p, g)
        rows.append(row)
    table = pd.DataFrame(rows)
    best = {g: table.loc[table[g].idxmin(), "support"] for g in gammas}
    return best, table

In [10]:
GAMMAS = [0.0, round(gamma_min + 0.05, 2), 0.5, 1.0]    # 0.0 = BIC
labels = {g: ("BIC" if g == 0 else f"EBIC(gamma={g})") for g in GAMMAS}

best, table = select_model(Xs, y, GAMMAS)
print(f"Evaluated {len(table)} distinct candidate models\n")

names = np.array(cols, dtype=object)
for g in GAMMAS:
    s = sorted(best[g])
    print(f"{labels[g]:<18} k = {len(s):>2}: {', '.join(names[s])}\n")

Evaluated 140 distinct candidate models

BIC                k = 15: ConstructionShareGrowth, ind2001_Finance_sh, InitialEUAccession, MEP_ConPct, MEP_Turnout, NONEU_Migrantgrowth, ind2001_Mining_sh, MEP_UKIPPct, MEP_GreenPct, MEP_BNPPct, Total_EconomyEU_dependence, quall4plus_2001_sh, noqual_sh_growth, quall4plus_sh_growth, ind2001_Agriculture_sh

EBIC(gamma=0.38)   k = 15: ConstructionShareGrowth, ind2001_Finance_sh, InitialEUAccession, MEP_ConPct, MEP_Turnout, NONEU_Migrantgrowth, ind2001_Mining_sh, MEP_UKIPPct, MEP_GreenPct, MEP_BNPPct, Total_EconomyEU_dependence, quall4plus_2001_sh, noqual_sh_growth, quall4plus_sh_growth, ind2001_Agriculture_sh

EBIC(gamma=0.5)    k = 15: ConstructionShareGrowth, ind2001_Finance_sh, InitialEUAccession, MEP_ConPct, MEP_Turnout, NONEU_Migrantgrowth, ind2001_Mining_sh, MEP_UKIPPct, MEP_GreenPct, MEP_BNPPct, Total_EconomyEU_dependence, quall4plus_2001_sh, noqual_sh_growth, quall4plus_sh_growth, ind2001_Agriculture_sh

EBIC(gamma=1.0)    k = 15: Construc

In [11]:
# Which covariates are chosen under which criterion
sel = pd.DataFrame({labels[g]: [c in set(names[sorted(best[g])]) for c in cols] for g in GAMMAS}, index=cols)
sel = sel[sel.any(axis=1)].sort_values(list(sel.columns)[::-1], ascending=False)
sel.replace({True: "x", False: ""})

,BIC,EBIC(gamma=0.38),EBIC(gamma=0.5),EBIC(gamma=1.0)
ConstructionShareGrowth,x,x,x,x
ind2001_Finance_sh,x,x,x,x
InitialEUAccession,x,x,x,x
MEP_ConPct,x,x,x,x
MEP_Turnout,x,x,x,x
NONEU_Migrantgrowth,x,x,x,x
ind2001_Mining_sh,x,x,x,x
MEP_UKIPPct,x,x,x,x
MEP_GreenPct,x,x,x,x
MEP_BNPPct,x,x,x,x


### 5. Stability of the selected set under subsampling
With $n$ in the hundreds and strongly correlated covariates, the "true set" is only credible if it survives resampling.
This cell reruns the whole selection procedure on random 80% subsamples, drawn **without** replacement, and reports how often each covariate is selected.
(A bootstrap with replacement duplicates rows, which makes spurious covariates look better and inflates the selected model size.)

In [12]:
GAMMA_FINAL = GAMMAS[1]       # smallest gamma above the Chen & Chen threshold; change if you prefer
B = 100
FRAC = 0.8
rng = np.random.default_rng(0)

counts = np.zeros(p)
sizes = []
for b in range(B):
    idx = rng.choice(n, int(FRAC * n), replace=False)
    Xb = Xs[idx]
    sd = Xb.std(axis=0)
    ok = sd > 0                                           # a subsample can make a dummy constant
    Xb = (Xb[:, ok] - Xb[:, ok].mean(axis=0)) / sd[ok]
    best_b, _ = select_model(Xb, y[idx], [GAMMA_FINAL])
    chosen = np.flatnonzero(ok)[sorted(best_b[GAMMA_FINAL])]
    counts[chosen] += 1
    sizes.append(len(chosen))

stability = (pd.Series(counts / B, index=cols, name="selection_freq")
               .sort_values(ascending=False))
in_full = set(names[sorted(best[GAMMA_FINAL])])
stab = stability[stability > 0].to_frame()
stab["selected_on_full_data"] = ["x" if c in in_full else "" for c in stab.index]
print(f"Subsample model size: median {np.median(sizes):.0f}, IQR {np.percentile(sizes, 25):.0f}-{np.percentile(sizes, 75):.0f}")
stab.head(30)

Subsample model size: median 15, IQR 13-17


,selection_freq,selected_on_full_data
MEP_GreenPct,1.00,x
quall4plus_2001_sh,1.00,x
ind2001_Mining_sh,1.00,x
MEP_BNPPct,1.00,x
MEP_UKIPPct,1.00,x
ind2001_Agriculture_sh,0.90,x
noqual_sh_growth,0.85,x
NONEU_Migrantgrowth,0.79,x
quall4plus_sh_growth,0.70,x
InitialEUAccession,0.68,x
